# 03. 异常检测

职责2-异常检测：统计模型 + 机器学习，从数据中自动发现异常模式，将人工经验规则升级为算法方案。

涵盖：3σ/IQR/Z-score/ESD 统计方法、Isolation Forest/One-Class SVM/XGBoost、滚动统计/STL/CUSUM 时序检测。

## 1. 统计方法对比

In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from utils.data_loader import load_byd_monthly
from anomaly import StatisticalAnomalyDetector

series = load_byd_monthly().set_index('month')['sales']
for method in ['sigma', 'iqr', 'zscore', 'esd']:
    det = StatisticalAnomalyDetector(method).detect(series)
    print(f'{method:8s}: {int(det["is_anomaly"].sum())} anomalies')

## 2. 时序异常检测

In [ ]:
from anomaly import TimeSeriesAnomalyDetector
from utils.data_loader import prepare_all
daily = prepare_all()['daily_sales'].set_index('date')['sales']

# STL 残差异常
det = TimeSeriesAnomalyDetector('stl_residual', period=7, k=3).detect(daily)
print(f'STL 残差: {int(det["is_anomaly"].sum())} anomalies')

# CUSUM 均值漂移
cusum = TimeSeriesAnomalyDetector('cusum', target=daily.mean(), std=daily.std(), k=0.5, h=5).detect(daily)
print(f'CUSUM: {int(cusum["is_anomaly"].sum())} anomalies')

## 3. 机器学习异常检测（多维）

In [ ]:
from anomaly import MLAnomalyDetector
md = prepare_all()['multi_dim_sales']
X = pd.get_dummies(md[['model','region','channel']])
X['sales'] = md['sales'].values

det = MLAnomalyDetector('isolation_forest', contamination=0.03).fit_predict(X)
print(f'Isolation Forest: {int(det["is_anomaly"].sum())} anomalies')
print(det[det['is_anomaly']].head())

## 4. 人工规则 → 算法方案升级示例

传统人工规则：`春节月销量 * 0.6`、`车型切换月 * 0.4`。

算法化升级：用 STL 分解自动捕捉季节性，用 Isolation Forest 检测多维异常。

In [ ]:
# 对比：人工规则 vs 算法
rule_anomalies = md[md['month'].dt.month.isin([1,2])].index  # 人工：1-2月异常
print(f'人工规则检出: {len(rule_anomalies)} 条')

if_det = MLAnomalyDetector('isolation_forest', contamination=0.05).fit_predict(X)
print(f'算法检出: {int(if_det["is_anomaly"].sum())} 条')
print(f'算法额外发现非春节异常: {(if_det["is_anomaly"] & ~md.index.isin(rule_anomalies)).sum()} 条')

## 小结

- 统计方法适合单指标快速筛查
- STL/CUSUM 适合时序季节性与漂移检测
- Isolation Forest 可发现人工规则遗漏的多维异常模式